# 02 — Padrões de fraude

Investiga associação descritiva por tipo, valor e etapa temporal. Associação não implica causalidade; os resultados refletem somente PaySim sintético. Requer `data/raw/`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data.prepare_data import find_csv
csv_path = find_csv(ROOT / 'data' / 'raw')
df = pd.read_csv(csv_path)
print(csv_path, df.shape)


## Por tipo de transação

In [ ]:
by_type=df.groupby('type').agg(transacoes=('isFraud','size'),fraudes=('isFraud','sum'),taxa_fraude=('isFraud','mean'),valor_mediano=('amount','median')).sort_values('taxa_fraude',ascending=False)
display(by_type)
ax=(100*by_type['taxa_fraude']).plot(kind='bar',color='#277da1',title='Taxa de fraude por tipo')
ax.set_ylabel('Fraudes (%)'); plt.xticks(rotation=35); plt.tight_layout(); plt.show()


## Valor da transação

In [ ]:
df['faixa_valor']=pd.qcut(df['amount'].rank(method='first'),q=10,labels=False)+1
by_amount=df.groupby('faixa_valor',observed=True).agg(transacoes=('isFraud','size'),fraudes=('isFraud','sum'),taxa_fraude=('isFraud','mean'),valor_mediano=('amount','median'))
display(by_amount)


## Evolução por `step`

In [ ]:
by_step=df.groupby('step').agg(transacoes=('isFraud','size'),fraudes=('isFraud','sum'))
by_step['taxa_fraude']=by_step['fraudes']/by_step['transacoes']
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(by_step.index,by_step['taxa_fraude']*100,lw=1)
ax.set(xlabel='step sintético (não é data)',ylabel='Fraudes (%)',title='Taxa de fraude por etapa temporal'); plt.tight_layout(); plt.show()


## Salvaguardas de interpretação
As taxas são descritivas da base simulada. Não inferir causalidade, comportamento de clientes reais, perdas financeiras ou efetividade em produção. Os identificadores não devem ser usados como variáveis preditoras sem justificativa.
